# 02 - Ground Truth Generation

Use the LLM to generate evaluation questions for each cocktail. The resulting `question -> cocktail id` pairs are the ground truth for retrieval and RAG evaluation.

In [ ]:
import sys
sys.path.append('../cocktail_assistant')

from dotenv import load_dotenv
load_dotenv('../.env')

In [ ]:
from ingest import load_data

documents = load_data('../data/cocktails.csv')
len(documents)

In [ ]:
from pydantic import BaseModel
from openai import OpenAI
from evaluation_utils import llm_structured_retry

client = OpenAI()


class GroundTruthQuestions(BaseModel):
    questions: list[str]


gt_instructions = """
You generate evaluation questions for a cocktail retrieval system.
Given one cocktail record, produce 5 short, natural questions a user might
ask that this specific cocktail answers. Vary the phrasing: some about the
name, some about ingredients, some about how to make it. Do not mention the
cocktail name in every question.
""".strip()

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress


def generate_for_doc(doc):
    prompt = (
        f"Name: {doc['name']}\n"
        f"Category: {doc['category']}\n"
        f"Ingredients: {doc['ingredients']}\n"
        f"Instructions: {doc['instructions']}"
    )
    result, _usage = llm_structured_retry(
        client, gt_instructions, prompt, GroundTruthQuestions
    )
    return [
        {"id": doc["id"], "name": doc["name"], "question": q}
        for q in result.questions
    ]


sample = documents[:50]  # keep the eval set small and cheap

with ThreadPoolExecutor(max_workers=4) as pool:
    nested = map_progress(pool, sample, generate_for_doc)

rows = [r for group in nested for r in group]
len(rows)

In [ ]:
import pandas as pd

gt_df = pd.DataFrame(rows)
gt_df.head()

In [ ]:
gt_df.to_csv('../data/ground_truth.csv', index=False)
gt_df.shape